In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

In [ ]:

arquivo_excel = (
    Path.home() # encontra a pasta no computador
    /"Documentos"/"Desafio GC"/"Python"/"DadosBrutos"/"resultados_cliente.xlsx"
)

# vê se o caminho e o arquivo existem
if arquivo_excel.is_file():
    print("\nArquivo encontrado")
else:
    print("\nArquivo não encontrado")


Arquivo não encontrado


In [4]:
# abre o arquivo para consultar os nomes das abas
with pd.ExcelFile(arquivo_excel, engine="openpyxl") as excel: # with fecha o arquivo no fim da execução

    # guarda os nomes encontrados em uma lista
    nomes_abas = excel.sheet_names

print("Abas encontradas:")
print(nomes_abas)


abas_esperadas = ["Patrocinador", "Mídia", "Pesquisa"]

for nome in abas_esperadas:

    if nome not in nomes_abas:
        print(f"\nA aba {nome} não foi encontrada")

print("\nAs abas necessárias foram encontradas")

FileNotFoundError: [Errno 2] No such file or directory: '/home/scherquy/Documentos/Desafio GC/Python/Dados/resultados_cliente.xlsx'

In [5]:
# lê cada uma das abas e guarda em uma variável
patrocinador = pd.read_excel(arquivo_excel, sheet_name="Patrocinador", engine="openpyxl")

midia = pd.read_excel(arquivo_excel, sheet_name="Mídia", engine="openpyxl")

pesquisa = pd.read_excel(arquivo_excel, sheet_name="Pesquisa", engine="openpyxl")

FileNotFoundError: [Errno 2] No such file or directory: '/home/scherquy/Documentos/Desafio GC/Python/Dados/resultados_cliente.xlsx'

In [5]:
# mostra quantas linhas de dados tem na tabela
print("PATROCINADOR")
print("Quantidade de registros:", len(patrocinador))
display(patrocinador)

print("\nMÍDIA")
print("Quantidade de registros:", len(midia))
display(midia)

print("\nPESQUISA")
print("Quantidade de registros:", len(pesquisa))
display(pesquisa)

PATROCINADOR
Quantidade de registros: 1


,Cliente_ID,Cliente_Nome,Cota,Evento,Edição,Cenário,Público_Pesquisa,Total_Respostas
0,C001,Cliente Verde,Subcota,Planeta Atlântida,2026,Protótipo Fictício,Visitantes do Cliente Verde,200



MÍDIA
Quantidade de registros: 10


,Registro_ID,Cliente_ID,Data,Canal,Métrica,Números,Origem
0,M001,C001,2026-01-30,Instagram,Visualizações,80000,SIMULADO
1,M002,C001,2026-01-31,Instagram,Visualizações,110000,SIMULADO
2,M003,C001,2026-01-30,TikTok,Visualizações,60000,SIMULADO
3,M004,C001,2026-01-31,TikTok,Visualizações,90000,SIMULADO
4,M005,C001,2026-01-30,YouTube,Visualizações,30000,SIMULADO
5,M006,C001,2026-01-31,YouTube,Visualizações,50000,SIMULADO
6,M007,C001,2026-01-30,TikTok,Menções,180,SIMULADO
7,M008,C001,2026-01-31,TikTok,Menções,240,SIMULADO
8,M009,C001,2026-01-30,Imprensa,Citações,12,SIMULADO
9,M010,C001,2026-01-31,Imprensa,Citações,18,SIMULADO



PESQUISA
Quantidade de registros: 30


,Registro_ID,Cliente_ID,Tema,Pergunta_ID,Categoria,Quantidade,Base_Respostas,Tipo_Resultado,Origem
0,P001,C001,genero,Q01,Feminino,96,200,DISTRIBUICAO,SIMULADO
1,P002,C001,genero,Q01,Masculino,78,200,DISTRIBUICAO,SIMULADO
2,P003,C001,genero,Q01,Outros ou prefere nao informar,26,200,DISTRIBUICAO,SIMULADO
3,P004,C001,faixa_etaria,Q02,14 a 15 anos,4,200,DISTRIBUICAO,SIMULADO
4,P005,C001,faixa_etaria,Q02,16 a 17 anos,20,200,DISTRIBUICAO,SIMULADO
5,P006,C001,faixa_etaria,Q02,18 a 24 anos,50,200,DISTRIBUICAO,SIMULADO
6,P007,C001,faixa_etaria,Q02,25 a 34 anos,40,200,DISTRIBUICAO,SIMULADO
7,P008,C001,faixa_etaria,Q02,35 a 49 anos,60,200,DISTRIBUICAO,SIMULADO
8,P009,C001,faixa_etaria,Q02,50 a 59 anos,20,200,DISTRIBUICAO,SIMULADO
9,P010,C001,faixa_etaria,Q02,60 anos ou mais,6,200,DISTRIBUICAO,SIMULADO


In [6]:
import re
import unicodedata

dados_validados = False

# guarda os avisos que precisam de verificação
pendencias_limpeza = []

# guarda as alterações feitas
alteracoes = []

def padronizar_nome(texto):
    texto = texto.strip().lower() # tira os espaços do início/fim

    texto = unicodedata.normalize("NFKD", texto) # separa as letras dos acentos

    texto = texto.encode("ascii", "ignore").decode("ascii")     # tira os acentos

    texto = re.sub(r"[\s-]+", "_", texto) # troca espaços e hífens por um sublinhado

    return texto


# cria cópias dos dados que já estão na memória
tabelas_limpas = {
    "patrocinador": patrocinador.copy(),
    "midia": midia.copy(),
    "pesquisa": pesquisa.copy()
}


for nome, tabela in tabelas_limpas.items():
    # padroniza os nomes das colunas
    novos_nomes = []

    for coluna in tabela.columns:
        novo_nome = padronizar_nome(str(coluna))
        novos_nomes.append(novo_nome)

        if novo_nome != coluna:
            alteracoes.append(
                f"{nome}: coluna '{coluna}' renomeada para '{novo_nome}'."
            )

    # duas colunas diferentes podem ficar com o mesmo nome e não escolhemos uma delas automaticamente
    if len(novos_nomes) != len(set(novos_nomes)):
        raise ValueError(
            f"Verificação necessária em '{nome}': "
            "duas colunas ficaram com o mesmo nome após a padronização."
        )

    tabela.columns = novos_nomes

    # Remove espaços do início/fim dos textos.
    for coluna in tabela.columns:
        for indice, valor in tabela[coluna].items():

            if isinstance(valor, str):
                texto_limpo = valor.strip()

                if texto_limpo != valor:
                    tabela.at[indice, coluna] = texto_limpo

                    alteracoes.append(
                        f"{nome}, linha {indice + 2}, "
                        f"coluna '{coluna}': espaços removidos."
                    )

print("Limpeza concluída\n")

Limpeza concluída



In [7]:
# os da esquerda são para comparação
# os da direita são os que vão ser guardados
padroes = {
    "canal": {
        "instagram": "Instagram",
        "tiktok": "TikTok",
        "youtube": "YouTube",
        "imprensa": "Imprensa"
    },

        "metrica": {
        "visualizacoes": "visualizacoes",
        "visualizacoes_video": "visualizacoes_video",
        "mencoes": "mencoes",
        "citacoes": "citacoes"
    },

    "unidade": {
        "visualizacoes": "visualizacoes",
        "mencoes": "mencoes",
        "citacoes": "citacoes"
    },

    "tipo_resultado": {
        "distribuicao": "DISTRIBUICAO",
        "indicador": "INDICADOR",
        "multipla_escolha": "MULTIPLA_ESCOLHA"
    },

    "origem": {
        "simulado": "SIMULADO"
    }
}

for nome, tabela in tabelas_limpas.items():

    for coluna, opcoes in padroes.items():

        # nem todas as tabelas têm todas essas colunas
        if coluna in tabela.columns:

            for indice, valor in tabela[coluna].items():

                if isinstance(valor, str) and valor != "":

                    chave = padronizar_nome(valor)

                    if chave in opcoes:
                        valor_padronizado = opcoes[chave]

                        if valor != valor_padronizado:
                            tabela.at[indice, coluna] = valor_padronizado

                            alteracoes.append(
                                f"{nome}, linha {indice + 2}, "
                                f"'{coluna}': '{valor}' -> "
                                f"'{valor_padronizado}'."
                            )

                    else:
                        pendencias_limpeza.append(
                            f"{nome}, linha {indice + 2}, "
                            f"'{coluna}': valor desconhecido '{valor}'. "
                            "Verifique e informe uma opção válida."
                        )

print("Padronização das categorias concluída")

Padronização das categorias concluída


In [8]:
# essas colunas representam números inteiros
campos_numericos = {
    "patrocinador": ["edicao", "total_respostas"],
    "midia": ["numeros"],
    "pesquisa": ["quantidade", "base_respostas"]
}


for nome, colunas in campos_numericos.items():

    tabela = tabelas_limpas[nome]

    for coluna in colunas:

        if coluna in tabela.columns:

            for indice, valor in tabela[coluna].items():

                if isinstance(valor, str) and valor != "":

                    # só converte textos que representam inteiros
                    if re.fullmatch(r"[+-]?[0-9]+", valor):

                        tabela.at[indice, coluna] = int(valor)

                        alteracoes.append(
                            f"{nome}, linha {indice + 2}, "
                            f"'{coluna}': texto convertido para número."
                        )

                    else:
                        # "1.000" não decidimos se significa mil ou um com casas decimais
                        pendencias_limpeza.append(
                            f"{nome}, linha {indice + 2}, "
                            f"'{coluna}': confira o valor '{valor}'. "
                            "Informe um inteiro sem separadores."
                        )


# conferimos as datas da mídia
tabela_midia = tabelas_limpas["midia"]

if "data" in tabela_midia.columns:

    for indice, valor in tabela_midia["data"].items():

        if isinstance(valor, str) and valor != "":

            # tenta o formato brasileiro 30/01/2026
            data_convertida = pd.to_datetime(
                valor,
                format="%d/%m/%Y",
                errors="coerce"
            )

            # Se não funcionou, tenta ano-mês-dia: 2026-01-30
            if pd.isna(data_convertida):
                data_convertida = pd.to_datetime(
                    valor,
                    format="%Y-%m-%d",
                    errors="coerce"
                )

            if pd.isna(data_convertida):

                pendencias_limpeza.append(
                    f"midia, linha {indice + 2}: "
                    f"a data '{valor}' precisa de verificação humana."
                )

            else:

                tabela_midia.at[indice, "data"] = data_convertida

                alteracoes.append(
                    f"midia, linha {indice + 2}: "
                    "texto convertido para data."
                )

print("Padronização de números e datas concluída.")

Padronização de números e datas concluída.


In [9]:
print("ALTERAÇÕES AUTOMÁTICAS:\n")

if alteracoes:
    display(pd.DataFrame({"alteracao": alteracoes}))
else:
    print("Nenhuma alteração automática foi necessária.")


print("\nPENDÊNCIAS PARA VERIFICAÇÃO HUMANA:\n")

if pendencias_limpeza:
    display(pd.DataFrame({"problema": pendencias_limpeza}))
else:
    print("Nenhuma pendência encontrada durante a limpeza.")

ALTERAÇÕES AUTOMÁTICAS:



,alteracao
0,patrocinador: coluna 'Cliente_ID' renomeada pa...
1,patrocinador: coluna 'Cliente_Nome' renomeada ...
2,patrocinador: coluna 'Cota' renomeada para 'co...
3,patrocinador: coluna 'Evento' renomeada para '...
4,patrocinador: coluna 'Edição' renomeada para '...
5,patrocinador: coluna 'Cenário' renomeada para ...
6,patrocinador: coluna 'Público_Pesquisa' renome...
7,patrocinador: coluna 'Total_Respostas' renomea...
8,midia: coluna 'Registro_ID' renomeada para 're...
9,midia: coluna 'Cliente_ID' renomeada para 'cli...



PENDÊNCIAS PARA VERIFICAÇÃO HUMANA:

Nenhuma pendência encontrada durante a limpeza.


In [10]:
print("PATROCINADOR — APÓS A LIMPEZA")
display(tabelas_limpas["patrocinador"])

print("\nMÍDIA — APÓS A LIMPEZA")
display(tabelas_limpas["midia"])

print("\nPESQUISA — APÓS A LIMPEZA")
display(tabelas_limpas["pesquisa"])

PATROCINADOR — APÓS A LIMPEZA


,cliente_id,cliente_nome,cota,evento,edicao,cenario,publico_pesquisa,total_respostas
0,C001,Cliente Verde,Subcota,Planeta Atlântida,2026,Protótipo Fictício,Visitantes do Cliente Verde,200



MÍDIA — APÓS A LIMPEZA


,registro_id,cliente_id,data,canal,metrica,numeros,origem
0,M001,C001,2026-01-30,Instagram,visualizacoes,80000,SIMULADO
1,M002,C001,2026-01-31,Instagram,visualizacoes,110000,SIMULADO
2,M003,C001,2026-01-30,TikTok,visualizacoes,60000,SIMULADO
3,M004,C001,2026-01-31,TikTok,visualizacoes,90000,SIMULADO
4,M005,C001,2026-01-30,YouTube,visualizacoes,30000,SIMULADO
5,M006,C001,2026-01-31,YouTube,visualizacoes,50000,SIMULADO
6,M007,C001,2026-01-30,TikTok,mencoes,180,SIMULADO
7,M008,C001,2026-01-31,TikTok,mencoes,240,SIMULADO
8,M009,C001,2026-01-30,Imprensa,citacoes,12,SIMULADO
9,M010,C001,2026-01-31,Imprensa,citacoes,18,SIMULADO



PESQUISA — APÓS A LIMPEZA


,registro_id,cliente_id,tema,pergunta_id,categoria,quantidade,base_respostas,tipo_resultado,origem
0,P001,C001,genero,Q01,Feminino,96,200,DISTRIBUICAO,SIMULADO
1,P002,C001,genero,Q01,Masculino,78,200,DISTRIBUICAO,SIMULADO
2,P003,C001,genero,Q01,Outros ou prefere nao informar,26,200,DISTRIBUICAO,SIMULADO
3,P004,C001,faixa_etaria,Q02,14 a 15 anos,4,200,DISTRIBUICAO,SIMULADO
4,P005,C001,faixa_etaria,Q02,16 a 17 anos,20,200,DISTRIBUICAO,SIMULADO
5,P006,C001,faixa_etaria,Q02,18 a 24 anos,50,200,DISTRIBUICAO,SIMULADO
6,P007,C001,faixa_etaria,Q02,25 a 34 anos,40,200,DISTRIBUICAO,SIMULADO
7,P008,C001,faixa_etaria,Q02,35 a 49 anos,60,200,DISTRIBUICAO,SIMULADO
8,P009,C001,faixa_etaria,Q02,50 a 59 anos,20,200,DISTRIBUICAO,SIMULADO
9,P010,C001,faixa_etaria,Q02,60 anos ou mais,6,200,DISTRIBUICAO,SIMULADO


In [11]:
tabelas_exportacao = {}

for nome, tabela in tabelas_limpas.items():
    tabelas_exportacao[nome] = tabela.copy()

correcoes_textos = {
    "mencoes": "Menções",
    "citacoes": "Citações",
    "visualizacoes": "Visualizações",
    "visualizacoes_video": "Visualizações de vídeo",

    "genero": "Gênero",
    "faixa_etaria": "Faixa etária",
    "domicilio": "Domicílio",
    "recorrencia": "Recorrência",
    "setor": "Setor",
    "dias_participacao": "Dias de participação",
    "importancia_marcas": "Importância das marcas",
    "experiencia_ativacao": "Experiência na ativação",
    "inclusao_acessibilidade": "Inclusão e acessibilidade",
    "acoes_esg": "Ações de ESG",

    "Outros ou prefere nao informar": "Outros ou prefere não informar",
    "Primeira participacao": "Primeira participação",
    "Ja participou antes": "Já participou antes",
    "Considera importante ou muito importante":
        "Considera importante ou muito importante",
    "Afirmou ter se divertido na ativacao":
        "Afirmou ter se divertido na ativação",
    "Considera a experiencia inclusiva e acessivel":
        "Considera a experiência inclusiva e acessível",

    "Hidratacao gratuita": "Hidratação gratuita",
    "Copos reutilizaveis": "Copos reutilizáveis",
    "Destinacao de residuos": "Destinação de resíduos",
    "Interpretes de Libras": "Intérpretes de Libras",
    "Alimentacao inclusiva": "Alimentação inclusiva",
    "Materiais ecologicos": "Materiais ecológicos",

    "DISTRIBUICAO": "Distribuição",
    "INDICADOR": "Indicador",
    "MULTIPLA_ESCOLHA": "Múltipla escolha",
    "SIMULADO": "Simulado",
    "Prototipo ficticio": "Protótipo fictício",
    "Protótipo Fictício": "Protótipo fictício"
}

# percorre as cópias e substitui os textos conhecidos
for nome, tabela in tabelas_exportacao.items():
    tabelas_exportacao[nome] = tabela.replace(correcoes_textos)

print("Textos preparados para a exportação.")

titulos_colunas = {
    "cliente_id": "Código do cliente",
    "cliente_nome": "Nome do cliente",
    "cota": "Cota",
    "evento": "Evento",
    "edicao": "Edição",
    "cenario": "Cenário",
    "publico_pesquisa": "Público da pesquisa",
    "total_respostas": "Total de respostas",
    "total_respondentes": "Total de respondentes",
    "documento_referencia": "Documento de referência",

    "registro_id": "Código do registro",
    "data": "Data",
    "canal": "Canal",
    "metrica": "Métrica",
    "numeros": "Números",
    "valor": "Valor",
    "unidade": "Unidade",
    "origem": "Origem",

    "tema": "Tema",
    "pergunta_id": "Código da pergunta",
    "categoria": "Categoria",
    "quantidade": "Quantidade",
    "base_respostas": "Base de respostas",
    "tipo_resultado": "Tipo de resultado"
}

for nome, tabela in tabelas_exportacao.items():
    tabelas_exportacao[nome] = tabela.rename(
        columns=titulos_colunas
    )

Textos preparados para a exportação.


In [ ]:
# Só permite exportar depois da validação.
if not dados_validados:
    raise ValueError(
        "Execute a validação e resolva as pendências antes de exportar."
    )

from datetime import datetime

tabelas_exportacao["midia"]["Data"] = pd.to_datetime(
    tabelas_exportacao["midia"]["Data"],
    errors="raise"
).dt.date

momento = datetime.now().strftime("%d%m%Y_%H%M%S")

pasta_saida = arquivo_excel.parent.parent / "DadosPadronizados"

pasta_saida.mkdir(parents=True, exist_ok=True)
 
arquivo_atualizado = (
    pasta_saida
    / f"resultados_cliente_padronizados_{momento}.xlsx"
)

with pd.ExcelWriter(
    arquivo_atualizado,
    engine="openpyxl",
    mode="x"
) as escritor:

    # salva os dados padronizados
    tabelas_exportacao["patrocinador"].to_excel(
        escritor,
        sheet_name="Patrocinador",
        index=False
    )

    tabelas_exportacao["midia"].to_excel(
        escritor,
        sheet_name="Mídia",
        index=False
    )

    tabelas_exportacao["pesquisa"].to_excel(
        escritor,
        sheet_name="Pesquisa",
        index=False
    )

    aba_midia = escritor.sheets["Mídia"]

    coluna_data = (
        tabelas_exportacao["midia"].columns.get_loc("Data") + 1
    )

    # formata as células de data
    for numero_linha in range(2, aba_midia.max_row + 1):

        celula = aba_midia.cell(
            row=numero_linha,
            column=coluna_data
        )

        celula.number_format = "dd/mm/yyyy"

print("Novo arquivo salvo em:")
print(arquivo_atualizado)

Novo arquivo salvo em:
/home/scherquy/Documentos/Desafio GC/Python/Dados/resultados_cliente_padronizados_16102026_143703.xlsx


In [13]:
import math

dados_validados = False

# avisos que a limpeza informou
erros_validacao = pendencias_limpeza.copy()

# define as colunas para o modelo atual
colunas_necessarias = {
    "patrocinador": [
        "cliente_id", "cliente_nome", "cota", "evento",
        "edicao", "cenario", "publico_pesquisa", "total_respostas"
    ],
    "midia": [
        "registro_id", "cliente_id", "data", "canal",
        "metrica", "numeros", "origem"
    ],
    "pesquisa": [
        "registro_id", "cliente_id", "tema", "pergunta_id",
        "categoria", "quantidade", "base_respostas",
        "tipo_resultado", "origem"
    ]
}

for nome, colunas in colunas_necessarias.items():

    tabela = tabelas_limpas[nome]

    # não aceita uma tabela sem registros neste protótipo
    if tabela.empty:
        erros_validacao.append(f"{nome}: tabela vazia.")

    # confere se cada coluna necessária existe
    for coluna in colunas:
        if coluna not in tabela.columns:
            erros_validacao.append(
                f"{nome}: falta a coluna '{coluna}'."
            )

if erros_validacao:

    for erro in erros_validacao:
        print("-", erro)

    raise ValueError(
        "Resolva as pendências de limpeza e estrutura antes de continuar."
    )

print("Estrutura das tabelas conferida")

Estrutura das tabelas conferida


In [14]:
for nome, tabela in tabelas_limpas.items():

    for coluna in colunas_necessarias[nome]:

        for indice, valor in tabela[coluna].items():

            linha_excel = indice + 2

            # acha blocos vazios e textos com espaços
            if pd.isna(valor) or str(valor).strip() == "":
                erros_validacao.append(
                    f"{nome}, linha {linha_excel}: "
                    f"o campo '{coluna}' está vazio."
                )

    # diz qual coluna identifica os registros da tabela
    if nome == "patrocinador":
        coluna_id = "cliente_id"
    else:
        coluna_id = "registro_id"

    # vê se algum identificador se repete
    if tabela[coluna_id].duplicated().any():
        erros_validacao.append(
            f"{nome}: há identificadores repetidos "
            f"na coluna '{coluna_id}'."
        )

print("Conferência de campos vazios e identificadores concluída")

Conferência de campos vazios e identificadores concluída


In [15]:
for nome, colunas in campos_numericos.items():

    tabela = tabelas_limpas[nome]

    for coluna in colunas:

        # tenta interpretar os valores como números e valores que não puderem ser interpretados viram NaN.
        numeros = pd.to_numeric(
            tabela[coluna],
            errors="coerce"
        )

        for indice, numero in numeros.items():

            linha_excel = indice + 2

            if pd.isna(numero) or not math.isfinite(numero):
                erros_validacao.append(
                    f"{nome}, linha {linha_excel}: "
                    f"'{coluna}' não contém um número válido"
                )

            elif numero < 0 or numero % 1 != 0:
                erros_validacao.append(
                    f"{nome}, linha {linha_excel}: "
                    f"'{coluna}' deve ser inteiro e não negativo"
                )

            elif coluna in ["total_respostas", "base_respostas"]:
                if numero == 0:
                    erros_validacao.append(
                        f"{nome}, linha {linha_excel}: "
                        f"'{coluna}' deve ser maior que zero"
                    )

        # guarda a coluna numérica na tabela da memória
        tabela[coluna] = numeros


# confere as datas tratadas pela limpeza
datas = pd.to_datetime(
    tabelas_limpas["midia"]["data"],
    format="%d/%m/%Y",
    errors="coerce"
)

if datas.isna().any():
    erros_validacao.append(
        "midia: há datas vazias, inválidas ou fora do formato"
    )
else:
    tabelas_limpas["midia"]["data"] = datas

print("Conferência de números e datas concluída")

Conferência de números e datas concluída


In [16]:
# só calcula os totais se os números anteriores forem válidos
if not erros_validacao:

    cadastro = tabelas_limpas["patrocinador"]
    dados_pesquisa = tabelas_limpas["pesquisa"]

    if len(cadastro) != 1:
        erros_validacao.append(
            "Este protótipo espera um patrocinador por arquivo."
        )

    else:
        # Pega as informações da única linha do cadastro.
        cliente = cadastro["cliente_id"].iloc[0]
        total_pessoas = cadastro["total_respostas"].iloc[0]

        # Todos os registros precisam pertencer ao cliente cadastrado.
        for nome in ["midia", "pesquisa"]:

            if (tabelas_limpas[nome]["cliente_id"] != cliente).any():
                erros_validacao.append(
                    f"{nome}: há registros de outro cliente."
                )

        # A mesma categoria não deve aparecer duas vezes
        # dentro da mesma pergunta para o mesmo cliente.
        categorias_repetidas = dados_pesquisa.duplicated(
            subset=["cliente_id", "pergunta_id", "categoria"]
        )

        if categorias_repetidas.any():
            erros_validacao.append(
                "pesquisa: há categorias repetidas na mesma pergunta."
            )

        # Reúne e confere os registros de cada pergunta.
        for pergunta, grupo in dados_pesquisa.groupby("pergunta_id"):

            # Cada pergunta precisa ter uma única base e um único tipo.
            bases_diferentes = grupo["base_respostas"].nunique()
            tipos_diferentes = grupo["tipo_resultado"].nunique()

            if bases_diferentes != 1 or tipos_diferentes != 1:
                erros_validacao.append(
                    f"{pergunta}: a base ou o tipo muda entre as linhas."
                )

            else:
                base = grupo["base_respostas"].iloc[0]
                tipo = grupo["tipo_resultado"].iloc[0]

                tipos_permitidos = [
                    "DISTRIBUICAO",
                    "INDICADOR",
                    "MULTIPLA_ESCOLHA"
                ]

                if tipo not in tipos_permitidos:
                    erros_validacao.append(
                        f"{pergunta}: tipo de resultado desconhecido."
                    )

                if base > total_pessoas:
                    erros_validacao.append(
                        f"{pergunta}: base maior que o total de pessoas."
                    )

                if (grupo["quantidade"] > base).any():
                    erros_validacao.append(
                        f"{pergunta}: uma quantidade ultrapassa a base."
                    )

                # Só distribuições completas precisam somar a base.
                if tipo == "DISTRIBUICAO":

                    total = grupo["quantidade"].sum()

                    if total != base:
                        erros_validacao.append(
                            f"{pergunta}: as quantidades somam {total}, "
                            f"mas a base é {base}."
                        )

print("Conferência do cliente e da pesquisa concluída.")

Conferência do cliente e da pesquisa concluída.


In [17]:
if erros_validacao:

    print("PENDÊNCIAS PARA REVISÃO HUMANA:")

    # Mostra os problemas em formato de tabela.
    display(
        pd.DataFrame({"problema": erros_validacao})
    )

    raise ValueError(
        "Os dados ainda precisam de revisão. "
        "Nenhum registro foi aprovado para envio ao banco."
    )

else:

    dados_validados = True

    print("Dados aprovados nas verificações do protótipo!")

    for nome, tabela in tabelas_limpas.items():
        print(f"{nome}: {len(tabela)} registro(s)")

Dados aprovados nas verificações do protótipo!
patrocinador: 1 registro(s)
midia: 10 registro(s)
pesquisa: 30 registro(s)
